In [ ]:
!pip install -q torch torchvision torchaudio scikit-learn pandas

from google.colab import drive
drive.mount("/content/drive")

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
)

import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

data_path = "/content/drive/MyDrive/SONARA/sonara_logmel_deltas_combined.npz"
print("NPZ exists:", os.path.exists(data_path))

data = np.load(data_path, allow_pickle=True)

X_train_old = data["X_train"]
y_train_3_old = data["y_train"]
X_val_old   = data["X_val"]
y_val_3_old = data["y_val"]
X_test_old  = data["X_test"]
y_test_3_old  = data["y_test"]
class_names_3 = data["class_names"].tolist()

print("3-class labels:", class_names_3)

# Reconstruct full pool
X_all_3 = np.concatenate([X_train_old, X_val_old, X_test_old], axis=0)
y_all_3 = np.concatenate([y_train_3_old, y_val_3_old, y_test_3_old], axis=0)

print("X_all_3:", X_all_3.shape)
print("y_all_3:", y_all_3.shape)
print("Class counts (3-class):", np.bincount(y_all_3))

Mounted at /content/drive
Device: cuda
NPZ exists: True
3-class labels: ['Normal', 'Respiratory Distress', 'Noise']
X_all_3: (6346, 3, 64, 400)
y_all_3: (6346,)
Class counts (3-class): [2139 2121 2086]


In [ ]:
# 3-class -> binary: Normal(0) & Noise(2) -> 0, Distress(1) -> 1
def to_binary(y3):
    y_bin = np.zeros_like(y3)
    y_bin[y3 == 1] = 1
    return y_bin

y_all_bin = to_binary(y_all_3)
print("Class counts (binary):", np.bincount(y_all_bin))

# New split: 70% train, 15% val, 15% test (approx)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_all_3, y_all_bin,
    test_size=0.3,
    stratify=y_all_bin,
    random_state=123,   # NEW SEED HERE (change this to try different splits)
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=123,
)

print("Train:", X_train.shape, np.bincount(y_train))
print("Val:  ", X_val.shape,   np.bincount(y_val))
print("Test: ", X_test.shape,  np.bincount(y_test))

binary_class_names = ["Non-distress", "Distress"]

Class counts (binary): [4225 2121]
Train: (4442, 3, 64, 400) [2957 1485]
Val:   (952, 3, 64, 400) [634 318]
Test:  (952, 3, 64, 400) [634 318]


In [ ]:
class LogMelBinaryDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = X
        self.y = y
        self.augment = augment

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx]  # (3, 64, 400)
        y = self.y[idx]
        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.long)
        if self.augment:
            x = self.apply_augment(x)
        return x, y

    def apply_augment(self, x):
        C, Freq, Time = x.shape
        if random.random() < 0.4:
            t_w = random.randint(5, max(5, Time // 8))
            t0 = random.randint(0, max(0, Time - t_w))
            x[:, :, t0:t0 + t_w] = x.mean()
        if random.random() < 0.4:
            f_h = random.randint(3, max(3, Freq // 8))
            f0 = random.randint(0, max(0, Freq - f_h))
            x[:, f0:f0 + f_h, :] = x.mean()
        if random.random() < 0.2:
            gain = 1.0 + 0.05 * torch.randn(1)
            x = x * gain
        return x

batch_size = 16

train_ds = LogMelBinaryDataset(X_train, y_train, augment=True)
val_ds   = LogMelBinaryDataset(X_val,   y_val,   augment=False)
test_ds  = LogMelBinaryDataset(X_test,  y_test,  augment=False)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                          num_workers=2, pin_memory=True)

In [ ]:
class CNN_BiLSTM(nn.Module):
    def __init__(self, num_classes=2, cnn_channels=64, lstm_hidden=128,
                 lstm_layers=1, dropout=0.3):
        super().__init__()
        self.conv1 = nn.Conv2d(3, cnn_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(cnn_channels)
        self.conv2 = nn.Conv2d(cnn_channels, cnn_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(cnn_channels)
        self.pool = nn.MaxPool2d(kernel_size=(2, 2))

        self.lstm_hidden = lstm_hidden
        self.lstm_layers = lstm_layers
        self.lstm = None
        self.dropout = nn.Dropout(dropout)
        self.fc = None
        self.num_classes = num_classes

    def _build_lstm_and_fc(self, x):
        B, C, Freq, Time = x.shape
        input_size = C * Freq
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=self.lstm_hidden,
            num_layers=self.lstm_layers,
            batch_first=True,
            bidirectional=True,
        ).to(x.device)
        self.fc = nn.Linear(self.lstm_hidden * 2, self.num_classes).to(x.device)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)

        if self.lstm is None or self.fc is None:
            self._build_lstm_and_fc(x)

        B, C, Freq, Time = x.shape
        x = x.permute(0, 3, 1, 2).contiguous()
        x = x.view(B, Time, C * Freq)

        x, _ = self.lstm(x)
        x = x[:, -1, :]
        x = self.dropout(x)
        x = self.fc(x)
        return x

model = CNN_BiLSTM().to(device)

# Initialize
xb_init, _ = next(iter(train_loader))
xb_init = xb_init.to(device)
with torch.no_grad():
    _ = model(xb_init)

total_params = sum(p.numel() for p in model.parameters())
print("Total params:", total_params)

Total params: 1221186


In [ ]:
train_counts_bin = np.bincount(y_train)
print("Binary train counts:", train_counts_bin)

class_weights = torch.tensor(
    train_counts_bin.mean() / train_counts_bin,
    dtype=torch.float32,
    device=device,
)
print("Binary class weights:", class_weights)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.02,
)

lr = 1e-4
weight_decay = 1e-4
num_epochs = 50
patience = 8

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=lr,
    weight_decay=weight_decay,
)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
)

best_val_loss = float("inf")
best_val_acc = 0.0
epochs_no_improve = 0

best_model_path = "/content/best_cnn_bilstm_sonara_logmel_binary_resplit.pt"

Binary train counts: [2957 1485]
Binary class weights: tensor([0.7511, 1.4956], device='cuda:0')


In [ ]:
for epoch in range(1, num_epochs + 1):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == yb).sum().item()
        train_total += yb.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            loss = criterion(logits, yb)

            val_loss += loss.item() * xb.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == yb).sum().item()
            val_total += yb.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    scheduler.step(val_loss)

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_model_path)
        star = "*"
    else:
        epochs_no_improve += 1
        star = ""

    current_lr = optimizer.param_groups[0]["lr"]
    print(
        f"Epoch {epoch:03d}/{num_epochs} "
        f"- lr: {current_lr:.1e} "
        f"- loss: {train_loss:.4f} acc: {train_acc:.4f} "
        f"- val_loss: {val_loss:.4f} val_acc: {val_acc:.4f} {star}"
    )

    if epochs_no_improve >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

print(
    f"Training complete. Best val loss: {best_val_loss:.4f} "
    f"Best val acc: {best_val_acc:.4f}"
)

Epoch 001/50 - lr: 1.0e-04 - loss: 0.6306 acc: 0.6475 - val_loss: 0.5640 val_acc: 0.6859 *
Epoch 002/50 - lr: 1.0e-04 - loss: 0.5688 acc: 0.7355 - val_loss: 0.5324 val_acc: 0.7384 *
Epoch 003/50 - lr: 1.0e-04 - loss: 0.5405 acc: 0.7517 - val_loss: 0.4957 val_acc: 0.7878 *
Epoch 004/50 - lr: 1.0e-04 - loss: 0.5250 acc: 0.7623 - val_loss: 0.4977 val_acc: 0.7710 
Epoch 005/50 - lr: 1.0e-04 - loss: 0.5010 acc: 0.7719 - val_loss: 0.4944 val_acc: 0.7847 *
Epoch 006/50 - lr: 1.0e-04 - loss: 0.4759 acc: 0.7929 - val_loss: 0.4762 val_acc: 0.8046 *
Epoch 007/50 - lr: 1.0e-04 - loss: 0.4817 acc: 0.7886 - val_loss: 0.4571 val_acc: 0.8277 *
Epoch 008/50 - lr: 1.0e-04 - loss: 0.4720 acc: 0.7956 - val_loss: 0.4382 val_acc: 0.8193 *
Epoch 009/50 - lr: 1.0e-04 - loss: 0.4578 acc: 0.8030 - val_loss: 0.4380 val_acc: 0.8078 *
Epoch 010/50 - lr: 1.0e-04 - loss: 0.4392 acc: 0.8136 - val_loss: 0.4673 val_acc: 0.8372 
Epoch 011/50 - lr: 1.0e-04 - loss: 0.4388 acc: 0.8176 - val_loss: 0.4321 val_acc: 0.8225 *
E

In [ ]:
state_dict = torch.load(best_model_path, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()
print("Loaded best model from:", best_model_path)

def collect_probs(dataloader):
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for xb, yb in dataloader:
            xb = xb.to(device)
            logits = model(xb)
            probs = F.softmax(logits, dim=1)[:, 1]  # p(distress)
            all_probs.append(probs.cpu())
            all_labels.append(yb)
    all_probs = torch.cat(all_probs, dim=0).numpy()
    all_labels = torch.cat(all_labels, dim=0).numpy()
    return all_probs, all_labels

val_probs, val_labels = collect_probs(val_loader)
test_probs, test_labels = collect_probs(test_loader)

print("Val probs shape:", val_probs.shape)
print("Test probs shape:", test_probs.shape)

Loaded best model from: /content/best_cnn_bilstm_sonara_logmel_binary_resplit.pt
Val probs shape: (952,)
Test probs shape: (952,)


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# Coarse search
coarse_best_thr = 0.5
coarse_best_acc = 0.0

for thr in np.linspace(0.3, 0.7, 81):
    preds = (val_probs >= thr).astype(int)
    acc = accuracy_score(val_labels, preds)
    if acc > coarse_best_acc:
        coarse_best_acc = acc
        coarse_best_thr = thr

print("Coarse best thr:", coarse_best_thr, "val_acc:", coarse_best_acc)

# Fine search around coarse best
fine_low  = max(0.0, coarse_best_thr - 0.05)
fine_high = min(1.0, coarse_best_thr + 0.05)

fine_best_thr = coarse_best_thr
fine_best_acc = coarse_best_acc
fine_best_f1  = 0.0

for thr in np.linspace(fine_low, fine_high, 101):
    preds = (val_probs >= thr).astype(int)
    acc = accuracy_score(val_labels, preds)
    f1  = f1_score(val_labels, preds, average="binary")
    if acc > fine_best_acc or (acc == fine_best_acc and f1 > fine_best_f1):
        fine_best_acc = acc
        fine_best_thr = thr
        fine_best_f1  = f1

print("Fine best thr:", fine_best_thr)
print("Fine best val_acc:", fine_best_acc)
print("Fine best val F1:", fine_best_f1)

# Apply to test
best_thr = fine_best_thr

test_preds = (test_probs >= best_thr).astype(int)

test_acc = accuracy_score(test_labels, test_preds)
macro_f1 = f1_score(test_labels, test_preds, average="macro")
bin_f1 = f1_score(test_labels, test_preds, average="binary")
roc_auc = roc_auc_score(test_labels, test_probs)

print("Binary test accuracy (resplit + tuned thr):", test_acc)
print("Macro F1:", macro_f1)
print("Distress F1:", bin_f1)
print("ROC-AUC:", roc_auc)

print(
    "Classification report (resplit + tuned thr):\n",
    classification_report(
        test_labels,
        test_preds,
        target_names=binary_class_names,
    )
)

cm = confusion_matrix(test_labels, test_preds)
print("Confusion matrix:\n", cm)

out_df = pd.DataFrame({
    "true_label_bin": test_labels,
    "prob_distress": test_probs,
    "pred_label_bin": test_preds,
})
out_path = "/content/cnn_bilstm_sonara_logmel_binary_resplit_predictions.csv"
out_df.to_csv(out_path, index=False)
print("Saved predictions to:", out_path)

Coarse best thr: 0.5049999999999999 val_acc: 0.8792016806722689
Fine best thr: 0.5019999999999999
Fine best val_acc: 0.8792016806722689
Fine best val F1: 0.8233486943164362
Binary test accuracy (resplit + tuned thr): 0.8897058823529411
Macro F1: 0.8795045655908146
Distress F1: 0.8444444444444444
ROC-AUC: 0.9397506100827332
Classification report (resplit + tuned thr):
               precision    recall  f1-score   support

Non-distress       0.94      0.89      0.91       634
    Distress       0.80      0.90      0.84       318

    accuracy                           0.89       952
   macro avg       0.87      0.89      0.88       952
weighted avg       0.90      0.89      0.89       952

Confusion matrix:
 [[562  72]
 [ 33 285]]
Saved predictions to: /content/cnn_bilstm_sonara_logmel_binary_resplit_predictions.csv


In [ ]:
# 3-class -> binary: Normal(0) & Noise(2) -> 0, Distress(1) -> 1
def to_binary(y3):
    y_bin = np.zeros_like(y3)
    y_bin[y3 == 1] = 1
    return y_bin

y_all_bin = to_binary(y_all_3)
print("Class counts (binary):", np.bincount(y_all_bin))

binary_class_names = ["Non-distress", "Distress"]

class LogMelBinaryDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = X
        self.y = y
        self.augment = augment

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.long)
        if self.augment:
            x = self.apply_augment(x)
        return x, y

    def apply_augment(self, x):
        C, Freq, Time = x.shape
        if random.random() < 0.4:
            t_w = random.randint(5, max(5, Time // 8))
            t0 = random.randint(0, max(0, Time - t_w))
            x[:, :, t0:t0 + t_w] = x.mean()
        if random.random() < 0.4:
            f_h = random.randint(3, max(3, Freq // 8))
            f0 = random.randint(0, max(0, Freq - f_h))
            x[:, f0:f0 + f_h, :] = x.mean()
        if random.random() < 0.2:
            gain = 1.0 + 0.05 * torch.randn(1)
            x = x * gain
        return x

def make_loaders_for_seed(seed, test_size=0.3):
    X_train, X_temp, y_train, y_temp = train_test_split(
        X_all_3, y_all_bin,
        test_size=test_size,
        stratify=y_all_bin,
        random_state=seed,
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.5,
        stratify=y_temp,
        random_state=seed,
    )

    train_ds = LogMelBinaryDataset(X_train, y_train, augment=True)
    val_ds   = LogMelBinaryDataset(X_val,   y_val,   augment=False)
    test_ds  = LogMelBinaryDataset(X_test,  y_test,  augment=False)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False,
                              num_workers=2, pin_memory=True)

    return train_loader, val_loader, test_loader, (X_train, X_val, X_test, y_train, y_val, y_test)

Class counts (binary): [4225 2121]


In [ ]:
def train_and_eval_for_seed(seed):
    print("\n====================")
    print(f"Seed {seed}")
    print("====================")

    # Re-split data
    train_loader, val_loader, test_loader, splits = make_loaders_for_seed(seed)
    X_train, X_val, X_test, y_train, y_val, y_test = splits

    # Build model
    model = CNN_BiLSTM().to(device)

    # Initialize LSTM/FC
    xb_init, _ = next(iter(train_loader))
    xb_init = xb_init.to(device)
    with torch.no_grad():
        _ = model(xb_init)

    # Loss/optimizer/scheduler
    train_counts_bin = np.bincount(y_train)
    class_weights = torch.tensor(
        train_counts_bin.mean() / train_counts_bin,
        dtype=torch.float32,
        device=device,
    )

    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=0.02,
    )

    lr = 1e-4
    weight_decay = 1e-4
    num_epochs = 40
    patience = 6

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
    )

    best_val_loss = float("inf")
    best_state = None
    epochs_no_improve = 0

    # Train
    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * xb.size(0)
            preds = logits.argmax(dim=1)
            train_correct += (preds == yb).sum().item()
            train_total += yb.size(0)

        train_loss /= train_total
        train_acc = train_correct / train_total

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                logits = model(xb)
                loss = criterion(logits, yb)

                val_loss += loss.item() * xb.size(0)
                preds = logits.argmax(dim=1)
                val_correct += (preds == yb).sum().item()
                val_total += yb.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        scheduler.step(val_loss)

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = model.state_dict()
            epochs_no_improve = 0
            star = "*"
        else:
            epochs_no_improve += 1
            star = ""

        current_lr = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch {epoch:03d} - lr: {current_lr:.1e} "
            f"- loss: {train_loss:.4f} acc: {train_acc:.4f} "
            f"- val_loss: {val_loss:.4f} val_acc: {val_acc:.4f} {star}"
        )

        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    # Restore best state
    model.load_state_dict(best_state)
    model.eval()

    # Collect probs
    def collect_probs(dataloader):
        all_probs = []
        all_labels = []
        with torch.no_grad():
            for xb, yb in dataloader:
                xb = xb.to(device)
                logits = model(xb)
                probs = F.softmax(logits, dim=1)[:, 1]
                all_probs.append(probs.cpu())
                all_labels.append(yb)
        all_probs = torch.cat(all_probs, dim=0).numpy()
        all_labels = torch.cat(all_labels, dim=0).numpy()
        return all_probs, all_labels

    val_probs, val_labels = collect_probs(val_loader)
    test_probs, test_labels = collect_probs(test_loader)

    # Threshold tuning (coarse + fine)
    coarse_best_thr = 0.5
    coarse_best_acc = 0.0
    for thr in np.linspace(0.3, 0.7, 81):
        preds = (val_probs >= thr).astype(int)
        acc = accuracy_score(val_labels, preds)
        if acc > coarse_best_acc:
            coarse_best_acc = acc
            coarse_best_thr = thr

    fine_low = max(0.0, coarse_best_thr - 0.05)
    fine_high = min(1.0, coarse_best_thr + 0.05)
    fine_best_thr = coarse_best_thr
    fine_best_acc = coarse_best_acc
    fine_best_f1 = 0.0
    for thr in np.linspace(fine_low, fine_high, 101):
        preds = (val_probs >= thr).astype(int)
        acc = accuracy_score(val_labels, preds)
        f1 = f1_score(val_labels, preds, average="binary")
        if acc > fine_best_acc or (acc == fine_best_acc and f1 > fine_best_f1):
            fine_best_acc = acc
            fine_best_thr = thr
            fine_best_f1 = f1

    best_thr = fine_best_thr

    # Final test metrics
    test_preds = (test_probs >= best_thr).astype(int)
    test_acc = accuracy_score(test_labels, test_preds)
    macro_f1 = f1_score(test_labels, test_preds, average="macro")
    bin_f1 = f1_score(test_labels, test_preds, average="binary")
    roc_auc = roc_auc_score(test_labels, test_probs)

    print(f"\nSeed {seed} results:")
    print("  Best val_acc:", fine_best_acc)
    print("  Test accuracy:", test_acc)
    print("  Macro F1:", macro_f1)
    print("  Distress F1:", bin_f1)
    print("  ROC-AUC:", roc_auc)

    cm = confusion_matrix(test_labels, test_preds)
    print("  Confusion matrix:\n", cm)

    return {
        "seed": seed,
        "val_acc": fine_best_acc,
        "test_acc": test_acc,
        "macro_f1": macro_f1,
        "distress_f1": bin_f1,
        "roc_auc": roc_auc,
        "thr": best_thr,
        "cm": cm,
    }

In [ ]:
seeds_to_try = [7, 21, 42, 77, 99]

all_results = []

for s in seeds_to_try:
    res = train_and_eval_for_seed(s)
    all_results.append(res)

print("\nSummary:")
for r in all_results:
    print(
        f"Seed {r['seed']}: "
        f"test_acc={r['test_acc']:.4f}, "
        f"val_acc={r['val_acc']:.4f}, "
        f"ROC-AUC={r['roc_auc']:.4f}, "
        f"Distress F1={r['distress_f1']:.4f}, thr={r['thr']:.3f}"
    )


Seed 7
Epoch 001 - lr: 1.0e-04 - loss: 0.6284 acc: 0.6659 - val_loss: 0.5836 val_acc: 0.6239 *
Epoch 002 - lr: 1.0e-04 - loss: 0.5649 acc: 0.7242 - val_loss: 0.5075 val_acc: 0.7962 *
Epoch 003 - lr: 1.0e-04 - loss: 0.5322 acc: 0.7560 - val_loss: 0.4934 val_acc: 0.7836 *
Epoch 004 - lr: 1.0e-04 - loss: 0.5268 acc: 0.7636 - val_loss: 0.5515 val_acc: 0.7973 
Epoch 005 - lr: 1.0e-04 - loss: 0.4983 acc: 0.7875 - val_loss: 0.4596 val_acc: 0.7847 *
Epoch 006 - lr: 1.0e-04 - loss: 0.4947 acc: 0.7832 - val_loss: 0.4492 val_acc: 0.8235 *
Epoch 007 - lr: 1.0e-04 - loss: 0.4659 acc: 0.8017 - val_loss: 0.4466 val_acc: 0.8351 *
Epoch 008 - lr: 1.0e-04 - loss: 0.4617 acc: 0.8066 - val_loss: 0.4439 val_acc: 0.8319 *
Epoch 009 - lr: 1.0e-04 - loss: 0.4582 acc: 0.8093 - val_loss: 0.4263 val_acc: 0.8361 *
Epoch 010 - lr: 1.0e-04 - loss: 0.4376 acc: 0.8183 - val_loss: 0.4355 val_acc: 0.8424 
Epoch 011 - lr: 1.0e-04 - loss: 0.4351 acc: 0.8240 - val_loss: 0.4299 val_acc: 0.8466 
Epoch 012 - lr: 1.0e-04 - l

In [ ]:
model.eval()
torch.save(model, "/content/sonara_cnn_bilstm_full.pt")
print("Saved full model to /content/sonara_cnn_bilstm_full.pt")

Saved full model to /content/sonara_cnn_bilstm_full.pt
